# UC-CAT-1 — Provision a Tenant Catalog

**Persona:** Platform Administrator (sysadmin)

**Goal:** Provision a new regional FAO tenant catalog end-to-end:
- Create the catalog via the STAC API
- Verify it appears in the catalog list
- Appoint a catalog admin user and confirm role assignment
- All operations use sysadmin credentials

**Prerequisites:**
- Platform running with `CatalogModule` registered (priority > 10)
- PostgreSQL healthy and reachable
- Sysadmin JWT in `DYNASTORE_SYSADMIN_TOKEN`

**Memory ref:** `project_geoid_iam_facade_refactor` — IAM facade refactor DONE 2026-04-14; `IamProtocol` split,
framework-free authz core, 4 capability protocols.

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_SYSADMIN_TOKEN`  
**Optional:** `CATALOG_ADMIN_EMAIL` (defaults to `data-engineer@fao.org`)

In [ ]:
import os
import json
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["DYNASTORE_BASE_URL"]
SYSADMIN_TOKEN = os.environ["DYNASTORE_SYSADMIN_TOKEN"]
ADMIN_EMAIL = os.environ.get("CATALOG_ADMIN_EMAIL", "data-engineer@fao.org")

CATALOG_ID = f"fao-test-{uuid.uuid4().hex[:8]}"

sysadmin_headers = {
    "Authorization": f"Bearer {SYSADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=sysadmin_headers, timeout=30.0)
print(f"Connected to {BASE_URL}  catalog_id={CATALOG_ID}")

## Step 1 — Create catalog

POST a `STACCatalogRequest` to the STAC catalogs endpoint. The platform assigns an isolated
PostgreSQL schema for this catalog on first write. A 201 response confirms the schema has been
created and the catalog record is persisted in `pg_collection_metadata`.

In [ ]:
catalog_payload = {
    "id": CATALOG_ID,
    "type": "Catalog",
    "title": "FAO Horn of Africa — Test Catalog",
    "description": "Regional catalog for Horn of Africa drought monitoring (test instance).",
    "stac_version": "1.0.0",
}

r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
print(r.status_code, r.text[:300])
assert r.status_code == 201, f"Expected 201, got {r.status_code}: {r.text}"

catalog = r.json()
assert catalog["id"] == CATALOG_ID, "Returned catalog id does not match request"
print(f"Catalog created: {catalog['id']}")

In [ ]:
# Verify the catalog is retrievable by id
r = client.get(f"/stac/catalogs/{CATALOG_ID}")
print(r.status_code, r.text[:300])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

fetched = r.json()
assert fetched["id"] == CATALOG_ID, "Fetched catalog id mismatch"
assert fetched["title"] == catalog_payload["title"], "Fetched catalog title mismatch"
print(f"Catalog verified: {fetched['id']}  title='{fetched['title']}'")

## Step 2 — Grant admin role to catalog admin user

After catalog creation, appoint a data engineer as the catalog administrator.
The `RoleGrantRequest` posts a role assignment to the IAM facade. Role `admin` gives
full read/write access within the catalog scope but cannot modify catalog-level IAM or
platform-global settings.

In [ ]:
role_payload = {"roles": ["admin"]}

r = client.post(
    f"/admin/users/{ADMIN_EMAIL}/roles/catalogs/{CATALOG_ID}",
    content=json.dumps(role_payload),
)
print(r.status_code, r.text[:300])
assert r.status_code == 201, f"Expected 201, got {r.status_code}: {r.text}"
print(f"Role 'admin' granted to {ADMIN_EMAIL} on catalog {CATALOG_ID}")

In [ ]:
# Confirm the admin email appears in the catalog user list
r = client.get(f"/admin/catalogs/{CATALOG_ID}/users")
print(r.status_code, r.text[:500])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

users = r.json()
user_emails = [
    u.get("email") or u.get("principal_id") or u.get("id", "")
    for u in (users if isinstance(users, list) else users.get("users", []))
]
print(f"Catalog users: {user_emails}")
assert any(ADMIN_EMAIL in str(e) for e in user_emails), (
    f"{ADMIN_EMAIL} not found in catalog user list: {user_emails}"
)
print(f"Confirmed: {ADMIN_EMAIL} is a member of catalog {CATALOG_ID}")

## Edge cases

The following cells demonstrate expected error behaviour. They should be reviewed but
do not affect the catalog state set up above.

In [ ]:
# Duplicate catalog_id must return 409 Conflict
r = client.post("/stac/catalogs", content=json.dumps(catalog_payload))
print(r.status_code, r.text[:300])
assert r.status_code == 409, (
    f"Expected 409 Conflict on duplicate catalog_id, got {r.status_code}: {r.text}"
)
print("409 Conflict confirmed for duplicate catalog_id — correct behaviour.")

In [ ]:
# A non-sysadmin token must receive 403 Forbidden when creating a catalog
# Uses a deliberately invalid / low-privilege bearer token.
non_sysadmin_token = os.environ.get("DYNASTORE_ADMIN_TOKEN", "invalid-token")
limited_headers = {
    "Authorization": f"Bearer {non_sysadmin_token}",
    "Content-Type": "application/json",
}
limited_client = httpx.Client(base_url=BASE_URL, headers=limited_headers, timeout=30.0)

alt_payload = {**catalog_payload, "id": f"{CATALOG_ID}-alt"}
r = limited_client.post("/stac/catalogs", content=json.dumps(alt_payload))
print(r.status_code, r.text[:300])
assert r.status_code in (401, 403), (
    f"Expected 401 or 403 for non-sysadmin token, got {r.status_code}: {r.text}"
)
print(f"{r.status_code} confirmed for non-sysadmin token — access correctly denied.")
limited_client.close()

## Teardown

Force-delete the test catalog. The `force=true` query parameter drops all collections,
items, and the PostgreSQL schema for this catalog. Only run this cell after reviewing
the notebook output above.

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"})
print(r.status_code, r.text[:300])
assert r.status_code == 204, f"Expected 204, got {r.status_code}: {r.text}"
print(f"Catalog {CATALOG_ID} deleted successfully.")
client.close()